
> **Public reproducibility notebook.** This notebook contains the experiment logic used in the paper. Machine-specific paths have been replaced with repository-relative configuration, and saved outputs are intentionally cleared.

# Standard Forecasting Baselines vs. Future-Compatible Retrieval

This notebook compares the frozen final finance retriever

\[
\boxed{\text{CrossStock Future-Compatibility Retrieval}}
\]

with standard direct forecasting baselines under the same temporal protocol. The retriever uses stock-local context, \(M=100\), \(K=10\), and horizons \(H\in\{1,5,20\}\).

---

# 1. Fair forecasting target

The target is the same future cumulative log-return path used by retrieval:

\[
y_j=\log P_{t+j}-\log P_t,\qquad j=1,\ldots,H.
\]

# 2. Direct-model input

The input is a 20-day cumulative log-return trajectory anchored at the current price,

\[
x_i=\log P_{t-L+i}-\log P_t,
\]

so the final input value is \(x_L=0\). Past and future therefore use the same relative log-price coordinate system.

# 3. Baselines

**Non-learning:** Zero Return and Linear Trend Extrapolation.

**Direct learned models:** Linear, two-layer MLP, MLP + Local Context, DLinear, PatchTST, and iTransformer. The MLP + Local Context control uses the same six local features as the final retriever, testing whether those features alone explain the retrieval result. DLinear, PatchTST, and iTransformer use the Time-Series-Library implementations.

Because the input here is univariate, iTransformer is not evaluated in its typical multivariate inverted-token regime; this limitation should be considered when interpreting its result.

# 4. Retrieval comparator

The already-trained three-seed CrossStock Learned Local-only \(M=100\) retriever is reloaded and evaluated on the identical test queries. The retrieval model is not retuned for this comparison.

# 5. Training protocol

For each horizon and seed, neural direct models are selected using validation ForecastMSE with early stopping, then refit on the train+validation period for the selected number of epochs. Test data are evaluated only after model selection.

# 6. Main metrics

Primary metrics are ForecastMSE and TerminalMAE; Direction Accuracy is secondary. Query-level differences between a direct baseline and retrieval are assessed with a 20-date moving-block bootstrap.

# 7. Methodological limitation

The expanded finance universe is based on a current S&P 500 snapshot and Yahoo Finance data, so it is survivorship-biased. We interpret this experiment as large-universe predictive robustness rather than as a claim about investable historical performance.

> **Public repository version.** Paths are repository-relative by default.
> Set `WHM_DATA_ROOT` to use datasets stored elsewhere and `WHM_WORK_ROOT` to move generated caches/checkpoints outside the repository.
> Saved execution outputs were cleared intentionally so the notebook does not expose machine-specific paths or stale results.


In [ ]:
from pathlib import Path
import os

def _find_repo_root(start=None):
    """Locate the repository root from the current working directory."""
    start = Path(start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "experiments").exists():
            return candidate
    raise RuntimeError(
        "Repository root not found. Start Jupyter from inside the cloned "
        "which-histories-matter repository, or set WHM_REPO_ROOT."
    )

_env_repo = os.environ.get("WHM_REPO_ROOT")
REPO_ROOT = Path(_env_repo).expanduser().resolve() if _env_repo else _find_repo_root()
REPO_DATA_ROOT = Path(os.environ.get("WHM_DATA_ROOT", REPO_ROOT / "data")).expanduser().resolve()
REPO_WORK_ROOT = Path(os.environ.get("WHM_WORK_ROOT", REPO_ROOT / "_work")).expanduser().resolve()
REPO_WORK_ROOT.mkdir(parents=True, exist_ok=True)

print("Repository root:", REPO_ROOT)
print("Data root:", REPO_DATA_ROOT)
print("Work root:", REPO_WORK_ROOT)


In [ ]:

from pathlib import Path
from types import SimpleNamespace
import sys
import math
import random
import time
import warnings
import gc

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import (
    TensorDataset,
    DataLoader,
)

from tqdm.auto import tqdm

warnings.filterwarnings("ignore")

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

DATA_ROOT = REPO_WORK_ROOT / "finance_case"

MH_DIR = (
    DATA_ROOT /
    "results" /
    "expanded_multihorizon_final"
)

RAW_FILE = (
    DATA_ROOT /
    "raw" /
    "yahoo_sp500_current_2000_2025.parquet"
)

RESULT_DIR = (
    DATA_ROOT /
    "results" /
    "standard_forecasting_baselines"
)

CACHE_DIR = (
    RESULT_DIR /
    "cache"
)

MODEL_DIR = (
    RESULT_DIR /
    "models"
)

RESULT_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

CACHE_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True,
)

assert MH_DIR.exists(), (
    "Run expanded multi-horizon experiment first."
)

assert RAW_FILE.exists(), RAW_FILE

HORIZONS = [
    1,
    5,
    20,
]

SEEDS = [
    0,
    1,
    2,
]

SEQ_LEN = 20

TOP_M = 100
TOP_K = 10

RETURN_SCALE = 100.0

MAX_EPOCHS = 30
PATIENCE = 6

MAX_TRAIN_SAMPLES_PER_PHASE = 60000

BLOCK_LEN = 20
N_BOOT = 5000

FORCE_REBUILD_DIRECT_DATA = False
FORCE_RETRAIN = False

TSL_ROOT = Path(os.environ.get(
    "TSL_ROOT", REPO_ROOT / "external" / "Time-Series-Library"
)).expanduser().resolve()

print(
    "Device:",
    DEVICE
)

if torch.cuda.is_available():

    print(
        "GPU:",
        torch.cuda.get_device_name(
            0
        )
    )

    print(
        "GPU memory GB:",
        torch.cuda.get_device_properties(
            0
        ).total_memory /
        1024**3
    )

print(
    "Multi-horizon dir:",
    MH_DIR
)

print(
    "Output dir:",
    RESULT_DIR
)


## 1. Load Time-Series-Library forecasting models

DLinear, PatchTST, and iTransformer are imported from an existing Time-Series-Library installation. This keeps the comparison tied to the library implementations rather than local simplified reimplementations.


In [ ]:

assert TSL_ROOT.exists(), (
    f"Time-Series-Library not found: {TSL_ROOT}"
)

if str(
    TSL_ROOT
) not in sys.path:

    sys.path.insert(
        0,
        str(
            TSL_ROOT
        ),
    )

try:

    from models import (
        DLinear,
        PatchTST,
        iTransformer,
    )

    print(
        "Time-Series-Library imports: OK"
    )

except Exception as e:

    print(
        "Failed to import Time-Series-Library models."
    )

    print(
        "Error:",
        repr(
            e
        )
    )

    raise


## 2. Load horizon-specific windows generated in the previous experiment

In [ ]:

WINDOWS = {}

for H in HORIZONS:

    path = (
        MH_DIR /
        "cache" /
        f"windows_H{H}.parquet"
    )

    assert path.exists(), path

    df = pd.read_parquet(
        path
    )

    df[
        "EndDate"
    ] = pd.to_datetime(
        df[
            "EndDate"
        ]
    )

    df[
        "FutureEndDate"
    ] = pd.to_datetime(
        df[
            "FutureEndDate"
        ]
    )

    WINDOWS[
        H
    ] = df

    print(
        f"H={H}:",
        len(
            df
        ),
        "windows |",
        df[
            "Ticker"
        ].nunique(),
        "tickers",
    )


## 3. Load raw Yahoo prices

In [ ]:

raw = pd.read_parquet(
    RAW_FILE
)

raw[
    "Date"
] = pd.to_datetime(
    raw[
        "Date"
    ]
)

raw = (
    raw.sort_values(
        [
            "Ticker",
            "Date",
        ]
    )
    .reset_index(
        drop=True
    )
)

print(
    "Raw rows:",
    len(
        raw
    )
)

print(
    "Tickers:",
    raw[
        "Ticker"
    ].nunique()
)


## 4. Reconstruct the direct-forecast input trajectory

Construct

\[
x=[\log P_{t-19}-\log P_t,\ldots,\log P_t-\log P_t],
\]

whose final value is zero. This matches the coordinate system of the future cumulative path.


In [ ]:

def build_past_paths(
    df,
    raw_df,
    seq_len=20,
):
    """
    Returns X aligned exactly with df row order.
    X shape = [N, seq_len]
    """

    raw_groups = {
        str(
            ticker
        ):
        g.sort_values(
            "Date"
        ).reset_index(
            drop=True
        )

        for ticker, g
        in raw_df.groupby(
            "Ticker"
        )
    }

    X = np.full(
        (
            len(
                df
            ),
            seq_len,
        ),
        np.nan,
        dtype=np.float32,
    )

    df_groups = (
        df.groupby(
            "Ticker"
        ).indices
    )

    for ticker, row_ids in tqdm(
        df_groups.items(),
        total=len(
            df_groups
        ),
        desc="Build past paths",
    ):

        ticker = str(
            ticker
        )

        if ticker not in raw_groups:
            continue

        g = raw_groups[
            ticker
        ]

        dates = pd.DatetimeIndex(
            g[
                "Date"
            ]
        )

        close = g[
            "Close"
        ].to_numpy(
            dtype=np.float64
        )

        log_close = np.log(
            close
        )

        row_ids = np.asarray(
            row_ids,
            dtype=np.int64,
        )

        anchor_dates = pd.DatetimeIndex(
            df.iloc[
                row_ids
            ][
                "EndDate"
            ]
        )

        pos = dates.get_indexer(
            anchor_dates
        )

        for rid, p in zip(
            row_ids,
            pos,
        ):

            if (
                p < seq_len - 1
                or
                p < 0
            ):
                continue

            segment = log_close[
                p -
                seq_len +
                1:
                p +
                1
            ]

            if (
                len(
                    segment
                ) != seq_len
                or
                not np.isfinite(
                    segment
                ).all()
            ):
                continue

            X[
                rid
            ] = (
                segment -
                log_close[
                    p
                ]
            ).astype(
                np.float32
            )

    return X


DIRECT_X = {}

for H in HORIZONS:

    cache_path = (
        CACHE_DIR /
        f"past_path_H{H}_L{SEQ_LEN}.npy"
    )

    if (
        cache_path.exists()
        and not
        FORCE_REBUILD_DIRECT_DATA
    ):

        X = np.load(
            cache_path
        )

        print(
            f"Loaded past-path cache H={H}:",
            X.shape
        )

    else:

        X = build_past_paths(
            WINDOWS[
                H
            ],
            raw,
            seq_len=SEQ_LEN,
        )

        np.save(
            cache_path,
            X,
        )

        print(
            "Saved:",
            cache_path
        )

    assert len(
        X
    ) == len(
        WINDOWS[
            H
        ]
    )

    finite_rows = np.isfinite(
        X
    ).all(
        axis=1
    )

    print(
        f"H={H} finite input coverage:",
        finite_rows.mean()
    )

    assert finite_rows.all(), (
        f"H={H}: some past paths could not be reconstructed."
    )

    max_last = float(
        np.abs(
            X[
                :,
                -1
            ]
        ).max()
    )

    print(
        f"H={H} max |last input|:",
        max_last
    )

    assert max_last < 1e-7

    DIRECT_X[
        H
    ] = X


## 5. Temporal splits — identical to retrieval experiment

In [ ]:

TRAIN_MEMORY_CUTOFF = pd.Timestamp(
    "2009-12-31"
)

TRAIN_START = pd.Timestamp(
    "2010-01-01"
)

TRAIN_END = pd.Timestamp(
    "2014-12-31"
)

VAL_MEMORY_CUTOFF = pd.Timestamp(
    "2014-12-31"
)

VAL_START = pd.Timestamp(
    "2015-01-01"
)

VAL_END = pd.Timestamp(
    "2019-12-31"
)

TEST_MEMORY_CUTOFF = pd.Timestamp(
    "2019-12-31"
)

TEST_START = pd.Timestamp(
    "2020-01-01"
)

SPLIT_INDEX = {}

for H in HORIZONS:

    df = WINDOWS[
        H
    ]

    train_idx = np.where(
        (
            df[
                "EndDate"
            ] >=
            TRAIN_START
        ) &
        (
            df[
                "FutureEndDate"
            ] <=
            TRAIN_END
        )
    )[0]

    val_idx = np.where(
        (
            df[
                "EndDate"
            ] >=
            VAL_START
        ) &
        (
            df[
                "FutureEndDate"
            ] <=
            VAL_END
        )
    )[0]

    test_idx = np.where(
        df[
            "EndDate"
        ] >=
        TEST_START
    )[0]

    SPLIT_INDEX[
        H
    ] = {
        "train":
            train_idx,

        "val":
            val_idx,

        "test":
            test_idx,
    }

    print(
        f"H={H}:",
        "train",
        len(
            train_idx
        ),
        "| val",
        len(
            val_idx
        ),
        "| test",
        len(
            test_idx
        ),
    )


## 6. Build targets and local context arrays

In [ ]:

LOCAL_COLS = [
    "STK_RET_5",
    "STK_RET_20",
    "STK_RV_20",
    "STK_VOL_RATIO",
    "STK_DD_60",
    "REL_STRENGTH_20",
]

TARGET_Y = {}
LOCAL_Z = {}
LOCAL_SCALER = {}

for H in HORIZONS:

    df = WINDOWS[
        H
    ]

    Y = np.stack(
        df[
            "FuturePath"
        ].to_numpy()
    ).astype(
        np.float32
    )

    assert Y.shape[
        1
    ] == H

    TARGET_Y[
        H
    ] = Y

    train_idx = SPLIT_INDEX[
        H
    ][
        "train"
    ]

    z = df[
        LOCAL_COLS
    ].to_numpy(
        dtype=np.float32
    )

    z_train = z[
        train_idx
    ]

    med = np.nanmedian(
        z_train,
        axis=0,
    )

    q25 = np.nanpercentile(
        z_train,
        25,
        axis=0,
    )

    q75 = np.nanpercentile(
        z_train,
        75,
        axis=0,
    )

    iqr = q75 - q25

    iqr = np.where(
        iqr <
        1e-6,
        1.0,
        iqr,
    )

    z_scaled = (
        z -
        med
    ) / iqr

    z_scaled = np.clip(
        z_scaled,
        -8.0,
        8.0,
    ).astype(
        np.float32
    )

    assert np.isfinite(
        z_scaled
    ).all()

    LOCAL_Z[
        H
    ] = z_scaled

    LOCAL_SCALER[
        H
    ] = {
        "Median":
            med.astype(
                np.float32
            ),

        "IQR":
            iqr.astype(
                np.float32
            ),
    }



## 7. Scale trajectories for neural-network optimization

Log returns are numerically small, so training uses percentage-like scale:

\[
x^{train}=100x,
\qquad
y^{train}=100y
\]

Evaluation converts predictions back to the original log-return scale.

This is a constant linear rescaling and does not change the forecasting problem.


In [ ]:

X_SCALED = {
    H:
    (
        DIRECT_X[
            H
        ] *
        RETURN_SCALE
    ).astype(
        np.float32
    )

    for H in HORIZONS
}

Y_SCALED = {
    H:
    (
        TARGET_Y[
            H
        ] *
        RETURN_SCALE
    ).astype(
        np.float32
    )

    for H in HORIZONS
}


## 8. Deterministic baselines

In [ ]:

def zero_forecast(
    X,
    horizon,
):
    return np.zeros(
        (
            len(
                X
            ),
            horizon,
        ),
        dtype=np.float32,
    )


def trend_forecast(
    X,
    horizon,
):
    """
    Least-squares linear slope of the past path,
    extrapolated from the current origin (0).
    """

    X = np.asarray(
        X,
        dtype=np.float64,
    )

    L = X.shape[
        1
    ]

    t = np.arange(
        -L + 1,
        1,
        dtype=np.float64,
    )

    t_centered = (
        t -
        t.mean()
    )

    denom = np.sum(
        t_centered ** 2
    )

    x_centered = (
        X -
        X.mean(
            axis=1,
            keepdims=True,
        )
    )

    slope = (
        x_centered @
        t_centered
    ) / denom

    future_t = np.arange(
        1,
        horizon + 1,
        dtype=np.float64,
    )

    pred = (
        slope[
            :,
            None
        ] *
        future_t[
            None,
            :
        ]
    )

    return pred.astype(
        np.float32
    )


## 9. Direct neural models

In [ ]:

class LinearForecaster(
    nn.Module
):
    def __init__(
        self,
        seq_len,
        pred_len,
    ):
        super().__init__()

        self.linear = nn.Linear(
            seq_len,
            pred_len,
        )

    def forward(
        self,
        x,
        z=None,
    ):
        return self.linear(
            x
        )


class MLPForecaster(
    nn.Module
):
    def __init__(
        self,
        input_dim,
        pred_len,
    ):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                input_dim,
                128,
            ),
            nn.LayerNorm(
                128
            ),
            nn.GELU(),
            nn.Dropout(
                0.10
            ),

            nn.Linear(
                128,
                64,
            ),
            nn.GELU(),
            nn.Dropout(
                0.10
            ),

            nn.Linear(
                64,
                pred_len,
            ),
        )

    def forward(
        self,
        x,
        z=None,
    ):

        if z is not None:

            x = torch.cat(
                [
                    x,
                    z,
                ],
                dim=1,
            )

        return self.net(
            x
        )


class TSLForecastWrapper(
    nn.Module
):
    def __init__(
        self,
        model,
        pred_len,
    ):
        super().__init__()

        self.model = model
        self.pred_len = pred_len

    def forward(
        self,
        x,
        z=None,
    ):
        # [B,L] -> [B,L,1]
        x_enc = x[
            ...,
            None
        ]

        try:

            out = self.model(
                x_enc,
                None,
                None,
                None,
            )

        except TypeError:

            out = self.model(
                x_enc
            )

        if isinstance(
            out,
            tuple,
        ):

            out = out[
                0
            ]

        if out.ndim == 3:

            # Expected [B,H,C]
            out = out[
                :,
                -self.pred_len:,
                0
            ]

        elif out.ndim == 2:

            out = out[
                :,
                -self.pred_len:
            ]

        else:

            raise RuntimeError(
                f"Unexpected TSL output shape: {out.shape}"
            )

        return out


## 10. Time-Series-Library configuration

In [ ]:

def make_tsl_config(
    horizon,
):
    return SimpleNamespace(
        task_name=
            "long_term_forecast",

        seq_len=
            SEQ_LEN,

        label_len=
            0,

        pred_len=
            horizon,

        enc_in=
            1,

        dec_in=
            1,

        c_out=
            1,

        d_model=
            64,

        n_heads=
            4,

        e_layers=
            2,

        d_layers=
            1,

        d_ff=
            128,

        factor=
            1,

        dropout=
            0.10,

        fc_dropout=
            0.10,

        head_dropout=
            0.0,

        embed=
            "timeF",

        freq=
            "d",

        activation=
            "gelu",

        output_attention=
            False,

        moving_avg=
            5,

        individual=
            False,

        class_strategy=
            "projection",

        use_norm=
            False,

        distil=
            True,

        top_k=
            5,

        num_kernels=
            6,
    )


def build_model(
    model_name,
    horizon,
):
    if model_name == "Linear":

        return LinearForecaster(
            SEQ_LEN,
            horizon,
        )

    if model_name == "MLP":

        return MLPForecaster(
            SEQ_LEN,
            horizon,
        )

    if model_name == "MLP_Local":

        return MLPForecaster(
            SEQ_LEN +
            len(
                LOCAL_COLS
            ),
            horizon,
        )

    cfg = make_tsl_config(
        horizon
    )

    if model_name == "DLinear":

        base_model = DLinear.Model(
            cfg
        )

    elif model_name == "PatchTST":

        try:

            base_model = PatchTST.Model(
                cfg,
                patch_len=5,
                stride=2,
            )

        except TypeError:

            print(
                "PatchTST constructor does not accept "
                "patch_len/stride; using library defaults."
            )

            base_model = PatchTST.Model(
                cfg
            )

    elif model_name == "iTransformer":

        base_model = iTransformer.Model(
            cfg
        )

    else:

        raise ValueError(
            model_name
        )

    return TSLForecastWrapper(
        base_model,
        horizon,
    )


## 11. Smoke-test every model before long training

Pass a dummy batch through each model and verify the expected shape \([B,20]\rightarrow[B,H]\) before beginning the full training run.


In [ ]:

TRAINABLE_MODELS = [
    "Linear",
    "MLP",
    "MLP_Local",
    "DLinear",
    "PatchTST",
    "iTransformer",
]

smoke_rows = []

for H in HORIZONS:

    x_dummy = torch.randn(
        8,
        SEQ_LEN,
        device=DEVICE,
    )

    z_dummy = torch.randn(
        8,
        len(
            LOCAL_COLS
        ),
        device=DEVICE,
    )

    for model_name in TRAINABLE_MODELS:

        model = build_model(
            model_name,
            H,
        ).to(
            DEVICE
        )

        model.eval()

        with torch.no_grad():

            out = model(
                x_dummy,
                z_dummy
                if model_name ==
                "MLP_Local"
                else None,
            )

        passed = (
            tuple(
                out.shape
            ) ==
            (
                8,
                H,
            )
            and
            torch.isfinite(
                out
            ).all().item()
        )

        smoke_rows.append({
            "Horizon":
                H,

            "Model":
                model_name,

            "OutputShape":
                str(
                    tuple(
                        out.shape
                    )
                ),

            "Finite":
                bool(
                    torch.isfinite(
                        out
                    ).all()
                ),

            "Passed":
                bool(
                    passed
                ),
        })

        del model

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

smoke_table = pd.DataFrame(
    smoke_rows
)

display(
    smoke_table
)

assert smoke_table[
    "Passed"
].all(), (
    "At least one baseline model failed its smoke test."
)

smoke_table.to_csv(
    RESULT_DIR /
    "01_model_smoke_test.csv",
    index=False,
)


## 12. Dataset / loader helpers

In [ ]:

def make_tensor_dataset(
    H,
    indices,
):
    idx = np.asarray(
        indices,
        dtype=np.int64,
    )

    x = torch.tensor(
        X_SCALED[
            H
        ][
            idx
        ],
        dtype=torch.float32,
    )

    y = torch.tensor(
        Y_SCALED[
            H
        ][
            idx
        ],
        dtype=torch.float32,
    )

    z = torch.tensor(
        LOCAL_Z[
            H
        ][
            idx
        ],
        dtype=torch.float32,
    )

    return TensorDataset(
        x,
        z,
        y,
    )


def sample_indices(
    indices,
    max_n,
    seed,
):
    idx = np.asarray(
        indices,
        dtype=np.int64,
    )

    if len(
        idx
    ) <= max_n:

        return idx

    rng = np.random.default_rng(
        seed
    )

    return np.sort(
        rng.choice(
            idx,
            size=max_n,
            replace=False,
        )
    )


def model_batch_size(
    model_name,
):
    if model_name in [
        "PatchTST",
        "iTransformer",
    ]:

        return 512

    return 1024


def model_learning_rate(
    model_name,
):
    if model_name in [
        "PatchTST",
        "iTransformer",
    ]:

        return 5e-4

    return 1e-3


## 13. Prediction / metric helpers

In [ ]:

@torch.no_grad()
def predict_model(
    model,
    dataset,
    model_name,
    batch_size,
):
    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
        pin_memory=True,
    )

    model.eval()

    pred_list = []
    true_list = []

    for x, z, y in loader:

        x = x.to(
            DEVICE,
            non_blocking=True,
        )

        z = z.to(
            DEVICE,
            non_blocking=True,
        )

        pred = model(
            x,
            z
            if model_name ==
            "MLP_Local"
            else None,
        )

        pred_list.append(
            pred.cpu()
        )

        true_list.append(
            y
        )

    pred = (
        torch.cat(
            pred_list,
            dim=0,
        ).numpy() /
        RETURN_SCALE
    )

    true = (
        torch.cat(
            true_list,
            dim=0,
        ).numpy() /
        RETURN_SCALE
    )

    return (
        pred.astype(
            np.float32
        ),
        true.astype(
            np.float32
        ),
    )


def forecast_metric_df(
    pred,
    true,
):
    pred = np.asarray(
        pred,
        dtype=np.float64,
    )

    true = np.asarray(
        true,
        dtype=np.float64,
    )

    mse = (
        (
            pred -
            true
        ) ** 2
    ).mean(
        axis=1
    )

    terminal_mae = np.abs(
        pred[
            :,
            -1
        ] -
        true[
            :,
            -1
        ]
    )

    direction = (
        np.sign(
            pred[
                :,
                -1
            ]
        ) ==
        np.sign(
            true[
                :,
                -1
            ]
        )
    ).astype(
        np.float32
    )

    return pd.DataFrame({
        "ForecastMSE":
            mse.astype(
                np.float32
            ),

        "TerminalMAE":
            terminal_mae.astype(
                np.float32
            ),

        "DirectionCorrect":
            direction,
    })


def summarize_forecast_metrics(
    metric_df,
):
    return {
        "ForecastMSE":
            float(
                metric_df[
                    "ForecastMSE"
                ].mean()
            ),

        "TerminalMAE":
            float(
                metric_df[
                    "TerminalMAE"
                ].mean()
            ),

        "DirectionAcc":
            float(
                metric_df[
                    "DirectionCorrect"
                ].mean()
            ),
    }


## 14. Training helpers

In [ ]:

def set_seed(
    seed,
):
    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(
            seed
        )


def assert_model_finite(
    model,
    where="",
):
    for name, p in (
        model.named_parameters()
    ):

        assert torch.isfinite(
            p
        ).all(), (
            f"Non-finite parameter "
            f"{name} at {where}"
        )


def train_one_epoch(
    model,
    loader,
    model_name,
    optimizer,
):
    model.train()

    losses = []

    for x, z, y in loader:

        x = x.to(
            DEVICE,
            non_blocking=True,
        )

        z = z.to(
            DEVICE,
            non_blocking=True,
        )

        y = y.to(
            DEVICE,
            non_blocking=True,
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        pred = model(
            x,
            z
            if model_name ==
            "MLP_Local"
            else None,
        )

        loss = F.mse_loss(
            pred,
            y,
        )

        assert torch.isfinite(
            loss
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=5.0,
        )

        optimizer.step()

        losses.append(
            float(
                loss.item()
            )
        )

    return float(
        np.mean(
            losses
        )
    )


## 15. Phase A — train and select best epoch on validation ForecastMSE

In [ ]:

def train_phase_a(
    H,
    model_name,
    seed,
):
    set_seed(
        seed
    )

    train_idx = sample_indices(
        SPLIT_INDEX[
            H
        ][
            "train"
        ],
        MAX_TRAIN_SAMPLES_PER_PHASE,
        seed=(
            100000 +
            H *
            100 +
            seed
        ),
    )

    val_idx = SPLIT_INDEX[
        H
    ][
        "val"
    ]

    train_ds = make_tensor_dataset(
        H,
        train_idx,
    )

    val_ds = make_tensor_dataset(
        H,
        val_idx,
    )

    batch_size = model_batch_size(
        model_name
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
        pin_memory=True,
    )

    model = build_model(
        model_name,
        H,
    ).to(
        DEVICE
    )

    lr = model_learning_rate(
        model_name
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=1e-4,
    )

    scheduler = (
        torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode="min",
            factor=0.5,
            patience=2,
            min_lr=1e-5,
        )
    )

    best_epoch = None
    best_val_mse = float(
        "inf"
    )

    wait = 0
    history = []

    t_start = time.time()

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):

        train_loss = train_one_epoch(
            model,
            train_loader,
            model_name,
            optimizer,
        )

        pred_val, true_val = (
            predict_model(
                model,
                val_ds,
                model_name,
                batch_size,
            )
        )

        val_metrics = (
            forecast_metric_df(
                pred_val,
                true_val,
            )
        )

        val_summary = (
            summarize_forecast_metrics(
                val_metrics
            )
        )

        val_mse = (
            val_summary[
                "ForecastMSE"
            ]
        )

        scheduler.step(
            val_mse
        )

        assert_model_finite(
            model,
            where=(
                f"H={H} "
                f"{model_name} "
                f"seed={seed} "
                f"epoch={epoch}"
            ),
        )

        history.append({
            "Horizon":
                H,

            "Model":
                model_name,

            "Seed":
                seed,

            "Epoch":
                epoch,

            "TrainN":
                len(
                    train_idx
                ),

            "TrainLossScaled":
                train_loss,

            "ValForecastMSE":
                val_mse,

            "ValTerminalMAE":
                val_summary[
                    "TerminalMAE"
                ],

            "ValDirectionAcc":
                val_summary[
                    "DirectionAcc"
                ],

            "LR":
                optimizer.param_groups[
                    0
                ][
                    "lr"
                ],
        })

        if (
            val_mse <
            best_val_mse -
            1e-10
        ):

            best_val_mse = (
                val_mse
            )

            best_epoch = (
                epoch
            )

            wait = 0

        else:

            wait += 1

        if wait >= PATIENCE:
            break

    elapsed = (
        time.time() -
        t_start
    )

    return (
        best_epoch,
        best_val_mse,
        elapsed,
        pd.DataFrame(
            history
        ),
    )


## 16. Phase B — refit on Train + Validation

In [ ]:

def refit_model(
    H,
    model_name,
    seed,
    epochs,
):
    set_seed(
        seed
    )

    model = build_model(
        model_name,
        H,
    ).to(
        DEVICE
    )

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=model_learning_rate(
            model_name
        ),
        weight_decay=1e-4,
    )

    batch_size = model_batch_size(
        model_name
    )

    train_idx = sample_indices(
        SPLIT_INDEX[
            H
        ][
            "train"
        ],
        MAX_TRAIN_SAMPLES_PER_PHASE,
        seed=(
            200000 +
            H *
            100 +
            seed
        ),
    )

    val_idx = sample_indices(
        SPLIT_INDEX[
            H
        ][
            "val"
        ],
        MAX_TRAIN_SAMPLES_PER_PHASE,
        seed=(
            300000 +
            H *
            100 +
            seed
        ),
    )

    phase_specs = [
        train_idx,
        val_idx,
    ]

    t_start = time.time()

    for epoch in range(
        1,
        epochs + 1,
    ):

        order = [
            0,
            1,
        ]

        random.Random(
            H *
            10000 +
            seed *
            100 +
            epoch
        ).shuffle(
            order
        )

        for phase_id in order:

            idx = phase_specs[
                phase_id
            ]

            ds = make_tensor_dataset(
                H,
                idx,
            )

            loader = DataLoader(
                ds,
                batch_size=batch_size,
                shuffle=True,
                num_workers=0,
                pin_memory=True,
            )

            _ = train_one_epoch(
                model,
                loader,
                model_name,
                optimizer,
            )

            assert_model_finite(
                model,
                where=(
                    f"refit H={H} "
                    f"{model_name} "
                    f"seed={seed} "
                    f"epoch={epoch}"
                ),
            )

    elapsed = (
        time.time() -
        t_start
    )

    return (
        model,
        elapsed,
    )


## 17. Train all direct forecasting models

The complete direct-model comparison contains

\[
3\ \text{horizons}\times 6\ \text{models}\times 3\ \text{seeds}=54\ \text{training runs}.
\]

Existing checkpoints are reused automatically unless retraining is forced.


In [ ]:

DIRECT_RESULTS = {}
HISTORY_FRAMES = []
SEED_ROWS = []

for H in HORIZONS:

    test_idx = SPLIT_INDEX[
        H
    ][
        "test"
    ]

    test_ds = make_tensor_dataset(
        H,
        test_idx,
    )

    for model_name in TRAINABLE_MODELS:

        for seed in SEEDS:

            print(
                "\n",
                "=" * 100
            )

            print(
                f"H={H} | "
                f"{model_name} | "
                f"seed={seed}"
            )

            print(
                "=" * 100
            )

            ckpt_path = (
                MODEL_DIR /
                f"H{H}_{model_name}_seed{seed}.pt"
            )

            if (
                ckpt_path.exists()
                and not FORCE_RETRAIN
            ):

                ckpt = torch.load(
                    ckpt_path,
                    map_location="cpu",
                    weights_only=False,
                )

                model = build_model(
                    model_name,
                    H,
                ).to(
                    DEVICE
                )

                model.load_state_dict(
                    ckpt[
                        "StateDict"
                    ]
                )

                best_epoch = int(
                    ckpt[
                        "BestEpoch"
                    ]
                )

                best_val = float(
                    ckpt[
                        "BestValidationMSE"
                    ]
                )

                phase_a_time = float(
                    ckpt.get(
                        "PhaseATimeSec",
                        np.nan,
                    )
                )

                refit_time = float(
                    ckpt.get(
                        "RefitTimeSec",
                        np.nan,
                    )
                )

                print(
                    "Loaded checkpoint."
                )

            else:

                (
                    best_epoch,
                    best_val,
                    phase_a_time,
                    hist,
                ) = train_phase_a(
                    H,
                    model_name,
                    seed,
                )

                HISTORY_FRAMES.append(
                    hist
                )

                print(
                    "Best epoch:",
                    best_epoch,
                    "| Val ForecastMSE:",
                    best_val,
                )

                (
                    model,
                    refit_time,
                ) = refit_model(
                    H,
                    model_name,
                    seed,
                    best_epoch,
                )

                torch.save(
                    {
                        "Horizon":
                            H,

                        "Model":
                            model_name,

                        "Seed":
                            seed,

                        "BestEpoch":
                            best_epoch,

                        "BestValidationMSE":
                            best_val,

                        "PhaseATimeSec":
                            phase_a_time,

                        "RefitTimeSec":
                            refit_time,

                        "StateDict":
                            model.state_dict(),

                        "ReturnScale":
                            RETURN_SCALE,

                        "LocalCols":
                            LOCAL_COLS,

                        "LocalMedian":
                            LOCAL_SCALER[
                                H
                            ][
                                "Median"
                            ],

                        "LocalIQR":
                            LOCAL_SCALER[
                                H
                            ][
                                "IQR"
                            ],
                    },
                    ckpt_path,
                )

            assert_model_finite(
                model,
                where=(
                    f"final H={H} "
                    f"{model_name} "
                    f"seed={seed}"
                ),
            )

            batch_size = model_batch_size(
                model_name
            )

            pred, true = predict_model(
                model,
                test_ds,
                model_name,
                batch_size,
            )

            metrics = forecast_metric_df(
                pred,
                true,
            )

            params = sum(
                p.numel()
                for p
                in model.parameters()
            )

            key = (
                H,
                model_name,
                seed,
            )

            DIRECT_RESULTS[
                key
            ] = {
                "pred":
                    pred,

                "true":
                    true,

                "metrics":
                    metrics,

                "best_epoch":
                    best_epoch,

                "best_val":
                    best_val,

                "params":
                    params,
            }

            row = {
                "Horizon":
                    H,

                "Model":
                    model_name,

                "Seed":
                    seed,

                "BestEpoch":
                    best_epoch,

                "BestValidationMSE":
                    best_val,

                "Parameters":
                    int(
                        params
                    ),

                "PhaseATimeSec":
                    phase_a_time,

                "RefitTimeSec":
                    refit_time,
            }

            row.update(
                summarize_forecast_metrics(
                    metrics
                )
            )

            SEED_ROWS.append(
                row
            )

            del model

            gc.collect()

            if torch.cuda.is_available():
                torch.cuda.empty_cache()

if HISTORY_FRAMES:

    history_table = pd.concat(
        HISTORY_FRAMES,
        ignore_index=True,
    )

else:

    history_table = pd.DataFrame()

history_table.to_csv(
    RESULT_DIR /
    "02_training_history.csv",
    index=False,
)

seed_table = pd.DataFrame(
    SEED_ROWS
)

display(
    seed_table
)

seed_table.to_csv(
    RESULT_DIR /
    "03_direct_seed_results.csv",
    index=False,
)


## 18. Evaluate Zero and Trend baselines

In [ ]:

DETERMINISTIC_RESULTS = {}

for H in HORIZONS:

    test_idx = SPLIT_INDEX[
        H
    ][
        "test"
    ]

    X_test = DIRECT_X[
        H
    ][
        test_idx
    ]

    Y_test = TARGET_Y[
        H
    ][
        test_idx
    ]

    for model_name, pred in [
        (
            "Zero",
            zero_forecast(
                X_test,
                H,
            ),
        ),
        (
            "Trend",
            trend_forecast(
                X_test,
                H,
            ),
        ),
    ]:

        metrics = forecast_metric_df(
            pred,
            Y_test,
        )

        DETERMINISTIC_RESULTS[
            (
                H,
                model_name,
            )
        ] = {
            "pred":
                pred,

            "metrics":
                metrics,
        }


## 19. Re-evaluate the frozen final retrieval models

This cell does not train a new retriever. It loads the previously saved

```text
expanded_multihorizon_final/models/H{H}_cross_LocalOnly_M100_seed{seed}.pt
```

checkpoints and reconstructs query-level results on the identical test set.


In [ ]:

class FutureCompatibilityReranker(
    nn.Module
):
    def __init__(
        self,
        context_dim,
        hidden_dim=128,
        dropout=0.10,
        initial_alpha=0.10,
    ):
        super().__init__()

        input_dim = (
            1 +
            4 *
            context_dim
        )

        self.mlp = nn.Sequential(
            nn.Linear(
                input_dim,
                hidden_dim,
            ),
            nn.LayerNorm(
                hidden_dim
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim,
                hidden_dim // 2,
            ),
            nn.GELU(),
            nn.Dropout(
                dropout
            ),

            nn.Linear(
                hidden_dim // 2,
                1,
            ),
        )

        raw_alpha = math.log(
            math.exp(
                initial_alpha
            ) -
            1.0
        )

        self.log_alpha_raw = nn.Parameter(
            torch.tensor(
                raw_alpha,
                dtype=torch.float32,
            )
        )

    @property
    def alpha(
        self
    ):
        return F.softplus(
            self.log_alpha_raw
        )

    def forward(
        self,
        pattern_score,
        query_context,
        cand_context,
    ):
        B, M, D = (
            cand_context.shape
        )

        q = (
            query_context[
                :, None, :
            ]
            .expand(
                -1,
                M,
                -1,
            )
        )

        diff = (
            q -
            cand_context
        )

        feat = torch.cat(
            [
                pattern_score[
                    ..., None
                ],
                q,
                cand_context,
                diff,
                diff.abs(),
            ],
            dim=-1,
        )

        delta = (
            self.mlp(
                feat
            )
            .squeeze(
                -1
            )
        )

        return (
            pattern_score +
            self.alpha *
            delta
        )


## 20. Retrieval evaluation helpers

In [ ]:

def robust_transform(
    df,
    cols,
    med,
    iqr,
):
    x = df[
        cols
    ].to_numpy(
        dtype=np.float32
    )

    x = (
        x -
        med
    ) / iqr

    x = np.clip(
        x,
        -8.0,
        8.0,
    )

    return x.astype(
        np.float32
    )


@torch.no_grad()
def evaluate_retrieval_seed(
    H,
    seed,
):
    ckpt_path = (
        MH_DIR /
        "models" /
        f"H{H}_cross_LocalOnly_M{TOP_M}_seed{seed}.pt"
    )

    pre_path = (
        MH_DIR /
        "cache" /
        f"cross_preselect_H{H}_M{TOP_M}.pt"
    )

    assert ckpt_path.exists(), ckpt_path
    assert pre_path.exists(), pre_path

    ckpt = torch.load(
        ckpt_path,
        map_location="cpu",
        weights_only=False,
    )

    pre = torch.load(
        pre_path,
        map_location="cpu",
        weights_only=False,
    )

    model = FutureCompatibilityReranker(
        context_dim=len(
            LOCAL_COLS
        )
    ).to(
        DEVICE
    )

    model.load_state_dict(
        ckpt[
            "StateDict"
        ]
    )

    model.eval()

    df = WINDOWS[
        H
    ]

    # Reproduce retrieval temporal split data frames.
    test_cand = df[
        df[
            "FutureEndDate"
        ] <=
        TEST_MEMORY_CUTOFF
    ].reset_index(
        drop=True
    )

    test_query = df[
        df[
            "EndDate"
        ] >=
        TEST_START
    ].reset_index(
        drop=True
    )

    # Direct test split order should match retrieval test_query order.
    direct_test_df = (
        df.iloc[
            SPLIT_INDEX[
                H
            ][
                "test"
            ]
        ]
        .reset_index(
            drop=True
        )
    )

    assert len(
        direct_test_df
    ) == len(
        test_query
    )

    assert (
        direct_test_df[
            [
                "Ticker",
                "EndDate",
            ]
        ].reset_index(
            drop=True
        )
        .equals(
            test_query[
                [
                    "Ticker",
                    "EndDate",
                ]
            ].reset_index(
                drop=True
            )
        )
    )

    med = np.asarray(
        ckpt[
            "Median"
        ],
        dtype=np.float32,
    )

    iqr = np.asarray(
        ckpt[
            "IQR"
        ],
        dtype=np.float32,
    )

    cols = list(
        ckpt[
            "ContextCols"
        ]
    )

    assert cols == LOCAL_COLS

    c_ctx = torch.tensor(
        robust_transform(
            test_cand,
            cols,
            med,
            iqr,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    q_ctx = torch.tensor(
        robust_transform(
            test_query,
            cols,
            med,
            iqr,
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    c_future = torch.tensor(
        np.stack(
            test_cand[
                "FuturePath"
            ].to_numpy()
        ).astype(
            np.float32
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    q_future = torch.tensor(
        np.stack(
            test_query[
                "FuturePath"
            ].to_numpy()
        ).astype(
            np.float32
        ),
        dtype=torch.float32,
        device=DEVICE,
    )

    test_idx = pre[
        "test_idx"
    ][
        :,
        :TOP_M
    ]

    test_score = pre[
        "test_score"
    ][
        :,
        :TOP_M
    ]

    assert len(
        test_idx
    ) == len(
        test_query
    )

    selected_list = []

    for start in range(
        0,
        len(
            test_query
        ),
        256,
    ):

        end = min(
            start +
            256,
            len(
                test_query
            ),
        )

        idx = test_idx[
            start:end
        ].to(
            DEVICE
        )

        ps = test_score[
            start:end
        ].to(
            DEVICE
        )

        qc = q_ctx[
            start:end
        ]

        cc = c_ctx[
            idx
        ]

        score = model(
            ps,
            qc,
            cc,
        )

        local = torch.topk(
            score,
            k=TOP_K,
            dim=1,
            largest=True,
        ).indices

        selected = torch.gather(
            idx,
            1,
            local,
        )

        selected_list.append(
            selected.cpu()
        )

    selected_idx = torch.cat(
        selected_list,
        dim=0,
    ).to(
        DEVICE
    )

    retrieved_future = c_future[
        selected_idx
    ]

    pred = retrieved_future.mean(
        dim=1
    )

    pred_np = pred.cpu().numpy()
    true_np = q_future.cpu().numpy()

    metrics = forecast_metric_df(
        pred_np,
        true_np,
    )

    return {
        "pred":
            pred_np.astype(
                np.float32
            ),

        "true":
            true_np.astype(
                np.float32
            ),

        "metrics":
            metrics,

        "alpha":
            float(
                model.alpha.item()
            ),
    }


## 21. Reconstruct 3-seed retrieval query-level results

In [ ]:

RETRIEVAL_RESULTS = {}

for H in HORIZONS:

    seed_results = []

    for seed in SEEDS:

        print(
            f"Retrieval H={H}, seed={seed}"
        )

        r = evaluate_retrieval_seed(
            H,
            seed,
        )

        seed_results.append(
            r
        )

    RETRIEVAL_RESULTS[
        H
    ] = seed_results


def mean_seed_metric_df(
    metric_frames,
):
    return pd.DataFrame({
        metric:
            np.stack(
                [
                    x[
                        metric
                    ].to_numpy()
                    for x
                    in metric_frames
                ],
                axis=0,
            ).mean(
                axis=0
            )

        for metric in [
            "ForecastMSE",
            "TerminalMAE",
            "DirectionCorrect",
        ]
    })


RETRIEVAL_MEAN_METRICS = {
    H:
    mean_seed_metric_df(
        [
            r[
                "metrics"
            ]
            for r
            in RETRIEVAL_RESULTS[
                H
            ]
        ]
    )

    for H in HORIZONS
}


## 22. Seed-average direct model query metrics

In [ ]:

DIRECT_MEAN_METRICS = {}

for H in HORIZONS:

    for model_name in TRAINABLE_MODELS:

        frames = [
            DIRECT_RESULTS[
                (
                    H,
                    model_name,
                    seed,
                )
            ][
                "metrics"
            ]

            for seed in SEEDS
        ]

        DIRECT_MEAN_METRICS[
            (
                H,
                model_name,
            )
        ] = mean_seed_metric_df(
            frames
        )


## 23. Main forecasting comparison table

In [ ]:

main_rows = []

for H in HORIZONS:

    # Deterministic
    for model_name in [
        "Zero",
        "Trend",
    ]:

        m = DETERMINISTIC_RESULTS[
            (
                H,
                model_name,
            )
        ][
            "metrics"
        ]

        row = {
            "Horizon":
                H,

            "Method":
                model_name,

            "Type":
                "Direct Forecast",
        }

        row.update(
            summarize_forecast_metrics(
                m
            )
        )

        main_rows.append(
            row
        )

    # Learned direct
    for model_name in TRAINABLE_MODELS:

        m = DIRECT_MEAN_METRICS[
            (
                H,
                model_name,
            )
        ]

        row = {
            "Horizon":
                H,

            "Method":
                model_name,

            "Type":
                "Direct Forecast",
        }

        row.update(
            summarize_forecast_metrics(
                m
            )
        )

        main_rows.append(
            row
        )

    # Retrieval
    m = RETRIEVAL_MEAN_METRICS[
        H
    ]

    row = {
        "Horizon":
            H,

        "Method":
            "CrossStock Future-Compatible Retrieval",

        "Type":
            "Retrieval",
    }

    row.update(
        summarize_forecast_metrics(
            m
        )
    )

    main_rows.append(
        row
    )

main_table = pd.DataFrame(
    main_rows
)

display(
    main_table
    .sort_values(
        [
            "Horizon",
            "ForecastMSE",
        ]
    )
)

main_table.to_csv(
    RESULT_DIR /
    "04_main_forecasting_comparison.csv",
    index=False,
)


## 24. Relative improvement of retrieval against every direct baseline

In [ ]:

improvement_rows = []

for H in HORIZONS:

    t = (
        main_table[
            main_table[
                "Horizon"
            ] ==
            H
        ]
        .set_index(
            "Method"
        )
    )

    retrieval_mse = float(
        t.loc[
            "CrossStock Future-Compatible Retrieval",
            "ForecastMSE",
        ]
    )

    for baseline in [
        "Zero",
        "Trend",
        *TRAINABLE_MODELS,
    ]:

        baseline_mse = float(
            t.loc[
                baseline,
                "ForecastMSE",
            ]
        )

        improvement_rows.append({
            "Horizon":
                H,

            "Baseline":
                baseline,

            "BaselineForecastMSE":
                baseline_mse,

            "RetrievalForecastMSE":
                retrieval_mse,

            "RetrievalImprovement_%":
                100.0 *
                (
                    baseline_mse -
                    retrieval_mse
                ) /
                baseline_mse,

            "RetrievalBetter":
                bool(
                    retrieval_mse <
                    baseline_mse
                ),
        })

improvement_table = pd.DataFrame(
    improvement_rows
)

display(
    improvement_table
)

improvement_table.to_csv(
    RESULT_DIR /
    "05_retrieval_relative_improvement.csv",
    index=False,
)


## 25. Query-level paired result files

In [ ]:

QUERY_LEVEL = {}

for H in HORIZONS:

    idx = SPLIT_INDEX[
        H
    ][
        "test"
    ]

    q = (
        WINDOWS[
            H
        ]
        .iloc[
            idx
        ][
            [
                "Ticker",
                "Sector",
                "EndDate",
            ]
        ]
        .reset_index(
            drop=True
        )
    )

    # Deterministic
    for model_name in [
        "Zero",
        "Trend",
    ]:

        m = DETERMINISTIC_RESULTS[
            (
                H,
                model_name,
            )
        ][
            "metrics"
        ]

        for metric in [
            "ForecastMSE",
            "TerminalMAE",
            "DirectionCorrect",
        ]:

            q[
                f"{model_name}_{metric}"
            ] = m[
                metric
            ].to_numpy()

    # Learned direct
    for model_name in TRAINABLE_MODELS:

        m = DIRECT_MEAN_METRICS[
            (
                H,
                model_name,
            )
        ]

        for metric in [
            "ForecastMSE",
            "TerminalMAE",
            "DirectionCorrect",
        ]:

            q[
                f"{model_name}_{metric}"
            ] = m[
                metric
            ].to_numpy()

    # Retrieval
    m = RETRIEVAL_MEAN_METRICS[
        H
    ]

    for metric in [
        "ForecastMSE",
        "TerminalMAE",
        "DirectionCorrect",
    ]:

        q[
            f"Retrieval_{metric}"
        ] = m[
            metric
        ].to_numpy()

    QUERY_LEVEL[
        H
    ] = q

    q.to_parquet(
        RESULT_DIR /
        f"06_query_level_H{H}.parquet",
        index=False,
    )


## 26. Moving-block bootstrap helpers

In [ ]:

def date_level_diff(
    dates,
    baseline,
    proposed,
):
    tmp = pd.DataFrame({
        "Date":
            pd.to_datetime(
                dates
            ),

        "Diff":
            np.asarray(
                baseline
            ) -
            np.asarray(
                proposed
            ),
    })

    return (
        tmp.groupby(
            "Date"
        )[
            "Diff"
        ]
        .mean()
        .sort_index()
    )


def moving_block_bootstrap(
    date_diff,
    block_len=20,
    n_boot=5000,
    seed=42,
):
    x = date_diff.to_numpy(
        dtype=np.float64
    )

    n = len(
        x
    )

    if n < block_len:

        raise ValueError(
            "Not enough dates."
        )

    rng = np.random.default_rng(
        seed
    )

    n_blocks = math.ceil(
        n /
        block_len
    )

    max_start = (
        n -
        block_len
    )

    boot = np.empty(
        n_boot,
        dtype=np.float64,
    )

    for b in range(
        n_boot
    ):

        pieces = []

        for _ in range(
            n_blocks
        ):

            s = rng.integers(
                0,
                max_start +
                1,
            )

            pieces.append(
                x[
                    s:
                    s +
                    block_len
                ]
            )

        sample = (
            np.concatenate(
                pieces
            )[
                :n
            ]
        )

        boot[
            b
        ] = (
            sample.mean()
        )

    return {
        "ObservedImprovement":
            float(
                x.mean()
            ),

        "CI_2.5%":
            float(
                np.quantile(
                    boot,
                    0.025,
                )
            ),

        "CI_97.5%":
            float(
                np.quantile(
                    boot,
                    0.975,
                )
            ),

        "P_improvement_gt_0":
            float(
                (
                    boot >
                    0
                ).mean()
            ),

        "N_dates":
            int(
                n
            ),

        "BlockLength":
            int(
                block_len
            ),
    }


## 27. Bootstrap retrieval vs. every direct baseline

A positive difference

\[
\text{baseline MSE}-\text{retrieval MSE}>0
\]

means retrieval is better. A 95% moving-block confidence interval with lower bound above zero is treated as statistically significant after accounting for temporal dependence.


In [ ]:

bootstrap_rows = []

for H in HORIZONS:

    q = QUERY_LEVEL[
        H
    ]

    for baseline in [
        "Zero",
        "Trend",
        *TRAINABLE_MODELS,
    ]:

        for metric in [
            "ForecastMSE",
            "TerminalMAE",
        ]:

            d = date_level_diff(
                q[
                    "EndDate"
                ],

                q[
                    f"{baseline}_{metric}"
                ],

                q[
                    f"Retrieval_{metric}"
                ],
            )

            r = moving_block_bootstrap(
                d,
                block_len=BLOCK_LEN,
                n_boot=N_BOOT,
                seed=(
                    H *
                    1000 +
                    len(
                        bootstrap_rows
                    )
                ),
            )

            r.update({
                "Horizon":
                    H,

                "Baseline":
                    baseline,

                "Metric":
                    metric,

                "RetrievalSignificantlyBetter":
                    bool(
                        r[
                            "CI_2.5%"
                        ] >
                        0
                    ),
            })

            bootstrap_rows.append(
                r
            )

bootstrap_table = pd.DataFrame(
    bootstrap_rows
)

display(
    bootstrap_table
)

bootstrap_table.to_csv(
    RESULT_DIR /
    "07_retrieval_vs_baselines_block_bootstrap.csv",
    index=False,
)


## 28. Parameter-count / efficiency table

In [ ]:

efficiency = (
    seed_table.groupby(
        [
            "Horizon",
            "Model",
        ]
    )
    .agg(
        Parameters=(
            "Parameters",
            "first",
        ),

        MeanBestEpoch=(
            "BestEpoch",
            "mean",
        ),

        MeanPhaseATimeSec=(
            "PhaseATimeSec",
            "mean",
        ),

        MeanRefitTimeSec=(
            "RefitTimeSec",
            "mean",
        ),
    )
    .reset_index()
)

display(
    efficiency
)

efficiency.to_csv(
    RESULT_DIR /
    "08_model_efficiency.csv",
    index=False,
)


## 29. Compact ranking table

In [ ]:

rank_rows = []

for H in HORIZONS:

    t = (
        main_table[
            main_table[
                "Horizon"
            ] ==
            H
        ]
        .sort_values(
            "ForecastMSE"
        )
        .reset_index(
            drop=True
        )
    )

    t[
        "Rank"
    ] = np.arange(
        1,
        len(
            t
        ) +
        1
    )

    rank_rows.append(
        t
    )

ranking_table = pd.concat(
    rank_rows,
    ignore_index=True,
)

display(
    ranking_table[
        [
            "Horizon",
            "Rank",
            "Method",
            "Type",
            "ForecastMSE",
            "TerminalMAE",
            "DirectionAcc",
        ]
    ]
)

ranking_table.to_csv(
    RESULT_DIR /
    "09_method_ranking.csv",
    index=False,
)


## 30. Final automatic decision summary

In [ ]:

decision_rows = []

for H in HORIZONS:

    t = (
        ranking_table[
            ranking_table[
                "Horizon"
            ] ==
            H
        ]
        .sort_values(
            "ForecastMSE"
        )
    )

    best_method = str(
        t.iloc[
            0
        ][
            "Method"
        ]
    )

    best_direct = (
        t[
            t[
                "Type"
            ] ==
            "Direct Forecast"
        ]
        .iloc[
            0
        ]
    )

    retrieval_row = (
        t[
            t[
                "Method"
            ] ==
            "CrossStock Future-Compatible Retrieval"
        ]
        .iloc[
            0
        ]
    )

    boot = bootstrap_table[
        (
            bootstrap_table[
                "Horizon"
            ] ==
            H
        ) &
        (
            bootstrap_table[
                "Baseline"
            ] ==
            best_direct[
                "Method"
            ]
        ) &
        (
            bootstrap_table[
                "Metric"
            ] ==
            "ForecastMSE"
        )
    ].iloc[
        0
    ]

    decision_rows.append({
        "Horizon":
            H,

        "BestOverallMethod":
            best_method,

        "BestDirectMethod":
            str(
                best_direct[
                    "Method"
                ]
            ),

        "BestDirectForecastMSE":
            float(
                best_direct[
                    "ForecastMSE"
                ]
            ),

        "RetrievalForecastMSE":
            float(
                retrieval_row[
                    "ForecastMSE"
                ]
            ),

        "RetrievalImprovementVsBestDirect_%":
            100.0 *
            (
                float(
                    best_direct[
                        "ForecastMSE"
                    ]
                ) -
                float(
                    retrieval_row[
                        "ForecastMSE"
                    ]
                )
            ) /
            float(
                best_direct[
                    "ForecastMSE"
                ]
            ),

        "BlockCI_Lower":
            float(
                boot[
                    "CI_2.5%"
                ]
            ),

        "BlockCI_Upper":
            float(
                boot[
                    "CI_97.5%"
                ]
            ),

        "RetrievalSignificantlyBetterThanBestDirect":
            bool(
                boot[
                    "CI_2.5%"
                ] >
                0
            ),
    })

decision_table = pd.DataFrame(
    decision_rows
)

display(
    decision_table
)

decision_table.to_csv(
    RESULT_DIR /
    "10_final_decision_summary.csv",
    index=False,
)


# Interpretation Guide

Default output directory:

```text
_work/finance_case/results/standard_forecasting_baselines/
```

Key files:

```text
03_direct_seed_results.csv
04_main_forecasting_comparison.csv
05_retrieval_relative_improvement.csv
07_retrieval_vs_baselines_block_bootstrap.csv
09_method_ranking.csv
10_final_decision_summary.csv
```

This experiment is intentionally diagnostic. If retrieval beats the best direct forecaster, it supports end-to-end predictive utility. If a direct model remains stronger, the result does not invalidate the retrieval-relevance claim: candidate selection and forecast aggregation are distinct objectives. The paper reports the latter outcome and therefore limits its main claim to historical relevance rather than forecasting SOTA.

`MLP_Local` is a particularly important control because it uses the same local observable features as the retriever. Direction Accuracy remains secondary; ForecastMSE and TerminalMAE are the primary finance metrics.
